# Data Generation Tutorial

This tutorial demonstrates how to generate synthetic eye tracking datasets using PyEtSimul. You'll learn to create variations for eye position, target position, and combined parameter changes to simulate different experimental conditions.

## Overview

PyEtSimul generates eye tracking data by:
1. Setting up hardware (eye, camera, light)
2. Defining parameter variations (what changes during data collection)
3. Running the data generation strategy to create synthetic measurements

This tutorial covers three main types of variations:
- **EyePositionVariation**: Move the eye in 3D space
- **TargetPositionVariation**: Move the gaze target in 3D space  
- **ComposedVariation**: Combine multiple parameter changes

## Setup

In [1]:
from pyetsimul.core import Eye, Camera, Light
from pyetsimul.types import Position3D, RotationMatrix
from pyetsimul.simulation import (
    DataGenerationStrategy,
    EyePositionVariation,
    TargetPositionVariation,
    ComposedVariation,
)

## Hardware Configuration

In [2]:
# Eye configuration
eye = Eye()
eye.set_rest_orientation(RotationMatrix([[1, 0, 0], [0, 0, 1], [0, 1, 0]], validate_handedness=False))
eye.position = Position3D(0.0, 550e-3, 350e-3)

eye.pprint()

Eye Anatomy Parameters:
+---------------------------------------+-------------------------+
| Parameter                             | Value                   |
+=======================================+=========================+
| Anterior corneal radius R_a (mm)      | 7.980                   |
+---------------------------------------+-------------------------+
| Posterior corneal radius R_p (mm)     | 6.220                   |
+---------------------------------------+-------------------------+
| Axial length L (mm)                   | 24.750                  |
+---------------------------------------+-------------------------+
| Cornea center to rotation center (mm) | 10.200                  |
+---------------------------------------+-------------------------+
| Thickness offset t_offset (mm)        | 1.150                   |
+---------------------------------------+-------------------------+
| Corneal depth d_c (mm)                | 3.540                   |
+-----------------------

In [3]:
# Camera configuration
camera = Camera(err=0.0, err_type="gaussian")
camera.orientation = RotationMatrix([[1, 0, 0], [0, 0, -1], [0, 1, 0]], validate_handedness=False)
camera.point_at(eye.position)

camera.pprint()

Camera Parameters:
+-------------------------+----------------------------------------+
| Parameter               | Value                                  |
+=========================+========================================+
| Focal length (px)       | 2880.0                                 |
+-------------------------+----------------------------------------+
| Resolution              | 1280 × 1024                            |
+-------------------------+----------------------------------------+
| Principal point (px)    | (640.0, 512.0)                         |
+-------------------------+----------------------------------------+
| Position (x,y,z) mm     | (0.0, 0.0, 0.0)                        |
+-------------------------+----------------------------------------+
| Distortion coefficients | k1=0.000, k2=0.000, p1=0.000, p2=0.000 |
+-------------------------+----------------------------------------+
| Measurement error       | 0.0000                                 |
+--------------

In [4]:
# Light configuration
light = Light(position=Position3D(200e-3, 0, 350e-3))

light.pprint()

Light Source Parameters:
+---------------------+--------------------------------+
| Parameter           | Value                          |
+=====================+================================+
| Position (x,y,z) mm | (200.000, 0.000, 350.000)      |
+---------------------+--------------------------------+
| Position (x,y,z) m  | (0.200000, 0.000000, 0.350000) |
+---------------------+--------------------------------+
| Light type          | Point source                   |
+---------------------+--------------------------------+


In [5]:
# Gaze target
gaze_target = Position3D(0.0, 0.0, 200e-3)

print(f"\nGaze Target: {gaze_target}")


Gaze Target: Position3D(x=0.0, y=0.0, z=0.2)


## 1. Eye Position Variation

Move the eye in XY plane around center position

In [6]:
# Move eye in XY plane around center position
eye_position_variation = EyePositionVariation(
    center=Position3D(0.0, 550e-3, 350e-3),
    dx=[-50e-3, 50e-3],       # ±50mm in X direction
    dy=[-50e-3, 50e-3],       # ±50mm in Y direction  
    dz=[0.0, 0.0],            # No Z movement
    grid_size=[16, 16, 1],    # 16x16 grid = 256 positions
)

print(f"Eye variation: {eye_position_variation.describe()}")
print(f"Total positions: {len(eye_position_variation)}")

Eye variation: eye position across 256 spatial locations
Total positions: 256


## 2. Target Position Variation

Move gaze target in XZ plane

In [7]:
# Move gaze target in XZ plane
target_position_variation = TargetPositionVariation(
    grid_center=Position3D(0, 0, 200e-3),
    dx=[-200e-3, 200e-3],     # ±200mm in X direction
    dy=[0.0, 0.0],            # No Y movement
    dz=[-150e-3, 150e-3],     # ±150mm in Z direction
    grid_size=[16, 1, 16],    # 16x1x16 grid = 256 positions
)

print(f"Target variation: {target_position_variation.describe()}")
print(f"Total positions: {len(target_position_variation)}")

Target variation: gaze targets across 256 screen positions
Total positions: 256


## 3. Composed Variation (Eye + Target Movement)

Combine eye and target movements for comprehensive dataset

In [8]:
# Combine eye and target movements

eye_target_movement_variation = ComposedVariation(
  [
      eye_position_variation,
      target_position_variation,
  ],
  "eye_target_movement",
)

print(f"Composed variation: {eye_target_movement_variation.describe()}")
print(f"Total combinations: {len(eye_target_movement_variation)}")

Composed variation: Combined: eye position across 256 spatial locations + gaze targets across 256 screen positions
Total combinations: 65536


## Generate Data

Now let's generate data for each variation type

In [9]:
def generate_data(variation_name, variation):
    """Generate data for a specific variation."""
    print(f"\n{'='*50}")
    print(f"Generating data for {variation_name}")
    print(f"{'='*50}")
    print(f"Variation type: {variation.__class__.__name__}")
    print(f"Total samples: {len(variation)}")
    
    strategy = DataGenerationStrategy(
        eyes=[eye],
        cameras=[camera],
        lights=[light],
        gaze_target=gaze_target,
        output_dir="outputs",
        experiment_name=variation_name,
    )
    
    dataset = strategy.execute(variation)
    print(f"Generated {dataset['total_measurements']} measurements")
    print(f"Saved to: outputs/{variation_name}_data.json")
    return dataset

### Generate Eye Position Data

In [10]:
# Generate data for eye position variation
eye_data = generate_data("eye_position", eye_position_variation)


Generating data for eye_position
Variation type: EyePositionVariation
Total samples: 256
Processing Camera 1/1, Eye 1/1...
Dataset saved to: outputs/eye_position_data.json
Generated 256 measurements
Saved to: outputs/eye_position_data.json


### Generate Target Position Data

In [11]:
# Generate data for target position variation
target_data = generate_data("target_position", target_position_variation)


Generating data for target_position
Variation type: TargetPositionVariation
Total samples: 256
Processing Camera 1/1, Eye 1/1...
Dataset saved to: outputs/target_position_data.json
Generated 256 measurements
Saved to: outputs/target_position_data.json


### Generate Composed Variation Data

In [12]:
# Generate data for composed variation
composed_data = generate_data("eye_target_movement", eye_target_movement_variation)


Generating data for eye_target_movement
Variation type: ComposedVariation
Total samples: 65536
Processing Camera 1/1, Eye 1/1...
Dataset saved to: outputs/eye_target_movement_data.json
Generated 256 measurements
Saved to: outputs/eye_target_movement_data.json


## Summary

This tutorial showed how to:

1. **Set up hardware configuration** - Eye, camera, and light positions
2. **Create parameter variations** - Eye position, target position, and composed variations
3. **Generate synthetic data** - Using DataGenerationStrategy to create measurements